In [ ]:
import argparse
import logging
import pathlib
import re
import math

import pandas as pd
import numpy as np
from Bio import SeqIO
from collections import Counter

import sgrna_subsample

import bioframe as bf
import bioframe.vis

In [ ]:
# Get all the seed sequences with their total number of hits
class Args():
    def __init__(self):
        self.genbank = None
        self.model_param = None
        self.file_find = None
        self.file_filter = None
        self.file_mismatch = None
        self.locus_tag = None
        self.step = None
        self.downstream = None
        self.off_target_seed = None
        self.pam_remove = None
        self.sgrna_remove = None
        self.downstream_remove = None
        self.gc_lower = None
        self.gc_upper = None
        self.offset_upper = None
        self.off_target_upper = None

args = Args()

args.genbank = headpath+'GCF_000009045.1_ASM904v1_genomic_Bsubtilis-168.gbff'
args.off_target_seed = 9

# Get all the seed sequences
off_target = list()
for m in re.finditer(r"(?=GG)", upper_seq):
    seed_sequence = upper_seq[
        (m.start() - 1 - args.off_target_seed):(m.start() - 1)
        ]
    off_target.append(seed_sequence)

for m in re.finditer(r"(?=GG)", lower_seq):
    seed_sequence = lower_seq[
        (m.start() - 1 - args.off_target_seed):(m.start() - 1)
    ]
    off_target.append(seed_sequence)

offtarget = Counter(off_target)

#### Get genes

In [ ]:
# Get gene df
# TODO: Assuming there is only one record. To be extended to multiple records

headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
path_genbank = headpath+'GCF_000009045.1_ASM904v1_genomic_Bsubtilis-168.gbff'
rec= SeqIO.read(path_genbank, "genbank")

# 'Upper' and 'Lower' strands of the genome
upper_seq = str(rec.seq)
lower_seq = str(rec.seq.reverse_complement())

if not rec.features:
    raise ValueError("No features found in the genbank file")

# Initialize a list with a length of the number of genes
list_genes = []
for feature in rec.features:
    if feature.type == "CDS":
        list_genes.append([feature.qualifiers.get("locus_tag", ["NA"])[0],
                           feature.qualifiers.get("gene", ["NA"])[0],
                           feature.location.strand,
                           feature.location.start,
                           feature.location.end,
                           feature.extract(rec.seq)])

# Convert the list to a dataframe
df_genes = pd.DataFrame(list_genes, columns=["locus_tag", "gene_name", "strand", "start", "end", "sequence"])
df_genes = (df_genes
            .assign(chrom = '0')
            )

#### Find matches across the genome


In [ ]:
# FILEPATH: /home/gutil/lag_paulsson-lab/library_design/2024-03-20_generate_sgrnas.ipynb
length_seed = 9

def find_off_targets(query_seed, upper_seq, lower_seq):

    assert len(query_seed) == length_seed

    list_seed_start_pos = []
    list_seed_end_pos = []
    list_spacer_start_pos = []
    list_spacers = []
    matches_upper_seq = []

    template_seq = upper_seq

    for m in re.finditer(query_seed+r'.GG', template_seq):


        # Get seed start and end (1-indexed)
        seed_start = m.start() + 1 # +1 for 0-based index
        seed_end = m.end() - 3 # 3 for the PAM
        list_seed_start_pos.append(seed_start)
        list_seed_end_pos.append(seed_end)

        # Get the spacer start (1-indexed)
        spacer_start = m.start()-(20-length_seed) + 1
        list_spacer_start_pos.append(spacer_start)
        # Get the spacer sequence
        list_spacers.append(template_seq[spacer_start - 1 : seed_end])
        matches_upper_seq.append(True)

    template_seq = lower_seq
    for m in re.finditer(query_seed+r'.GG', lower_seq):

        # Get seed start and end (1-indexed) in both rc and fw coordinates
        seed_start_rc_coords = m.start() + 1 # 1 for 0-based index
        seed_end_rc_coords = m.end() - 3 # 3 for the PAM
        seed_start = len(lower_seq) - seed_start_rc_coords + 1
        seed_end = len(lower_seq)  - seed_end_rc_coords + 1
        list_seed_start_pos.append(seed_start)
        list_seed_end_pos.append(seed_end)

        # Get the spacer start (1-indexed) in both rc and fw coordinates
        spacer_start_rc_coords = m.start()-(20-length_seed) + 1
        spacer_start = len(lower_seq) - spacer_start_rc_coords + 1
        list_spacer_start_pos.append(spacer_start)
        # Get the spacer sequence
        list_spacers.append(template_seq[spacer_start_rc_coords - 1 : seed_end_rc_coords])
        matches_upper_seq.append(False)
    
    n_targets = len(list_seed_start_pos)

    return query_seed, n_targets, list_seed_start_pos, list_seed_end_pos, list_spacer_start_pos, list_spacers, matches_upper_seq


In [ ]:
# query_seeds = ['TCCATCTTC', 'AGACTTTGC', 'GAAAGAGCT', 'AAAAGAAAA', 'AAAACGATT', 'TATCCACAG']
# query_seeds = ['TATCCACAG']
query_seeds = ['TATCTTTTT', 'TATCCACAG', 'GCGACATGT']

df_seeds = pd.DataFrame(index = np.arange(len(query_seeds)),
                    columns=['query_seed', 'n_targets', 'list_seed_start_pos', 'list_seed_end_pos', 'list_spacer_start_pos', 'list_spacers', 'matches_upper_seq'])

# Iterate over the query seeds
for i, query_seed in enumerate(query_seeds):
    # Call the function and fill the dataframe
    # Append the values to the dataframe
    df_seeds.loc[i, :] = find_off_targets(query_seed, upper_seq, lower_seq)
# Print the dataframe
df_seeds = (df_seeds.reset_index()
                    .rename(columns={"index": "seed_index"})
                    .explode(['list_seed_start_pos', 'list_seed_end_pos', 'list_spacer_start_pos', 'list_spacers', 'matches_upper_seq'])
                    .assign(start = lambda df_: df_.apply(lambda row: row['list_spacer_start_pos'] if row['matches_upper_seq'] else row['list_seed_end_pos'], axis=1),
                      end = lambda df_: df_.apply(lambda row: row['list_seed_end_pos'] if row['matches_upper_seq'] else row['list_spacer_start_pos'], axis=1),
                      chrom = '0')
)
df_seeds

#### Find the predicted efficacy of a gRNA

In [ ]:
# PREPROCESS LOOKUP TABLE
DNA_BASES = ["A", "T", "C", "G"]

lookup = pd.read_csv(headpath+'model_param_bsub.csv')
features = lookup["feature"].to_list()
weights = lookup["weight"].to_list()
lookuptable = []
for loc in range(20):
    for wt in range(len(DNA_BASES)):
        n_wt = DNA_BASES[wt]
        for mu in range(len(DNA_BASES)):
            n_mu = DNA_BASES[mu]
            if n_wt != n_mu:
                P1 = float(weights[features.index(str(loc))]) + float(
                    weights[features.index(n_wt + n_mu)]
                )
                lookuptable.append((loc, n_wt, n_mu, P1))
lookuptable = pd.DataFrame(lookuptable, columns=["loc", "wt", "mu", "P1"])

loc = lookuptable["loc"].to_list()
wt = lookuptable["wt"].to_list()
mu = lookuptable["mu"].to_list()
P1 = lookuptable["P1"].to_list()
intercept = float(weights[features.index("intercept")])
GC_weight = float(weights[features.index("GC_content")])

In [ ]:
lookuptable

## Map with gene annotations

## Format to bedframe

In [ ]:
bioframe.is_bedframe(df_genes), bioframe.is_bedframe(df_seeds)

In [ ]:
bioframe.overlap(df1 = df_seeds, df2 = df_genes, suffixes=('_seeds', '_genes'), how = 'left')

## Run for the controls

In [ ]:
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"

filename_controls = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
df_controls = (pd.read_excel(headpath+filename_controls,
                    sheet_name="relative fitness (bsu controls)")
                 # Remove row with nans
                 .rename(columns={'relative fitness (stddev)':'relative_fitness_std',
                                 'relative fitness (mean)':'relative_fitness_mean',
                                 'variant': 'mismatch_sgrna',
                                 'original': 'full_sgrna'}) # Rename column
                 .dropna(subset=['relative_fitness_mean'])
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.check_all_sites) == False] # Remove rows with restriction sites
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites
                 # Remove rows for which 'relative fitness (stdev)' is less than 0.02
               #   .loc[lambda df_: df_['relative_fitness_std'] < 0.02]
                 # Filter rows whose relative_fitness_mean is between 1 + delta and 1 - delta
               #   .loc[lambda df_: df_['relative_fitness_mean'].between(1-0.003, 1+0.003)] # TODO: Let delta be variable?
               #   .sample(n=100, replace=False) # Sample 100 controls
                 .assign(seed = lambda df_: df_['full_sgrna'].str[-9:])
                 .assign(category = 'control')
                 .astype({'category': 'category'})
                 .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))
                #  .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Check for BsaI and PvuI sites only
    )

In [ ]:
df_controls

In [ ]:
query_seeds = list(df_controls['seed'].str[-9:])

df = pd.DataFrame(index = np.arange(len(query_seeds)),
                    columns=['query_seed', 'n_targets', 'list_seed_start_pos', 'list_seed_end_pos', 'list_spacer_start_pos', 'list_spacers', 'matches_upper_seq'])

# Iterate over the query seeds
for i, query_seed in enumerate(query_seeds):
    # Call the function and fill the dataframe
    # Append the values to the dataframe
    df.loc[i, :] = find_off_targets(query_seed, upper_seq, lower_seq)
# Print the dataframe

In [ ]:
df_controls

In [ ]:
df2 = df_controls.merge(df, left_on='seed', right_on='query_seed')

In [ ]:
df2

In [ ]:
df2[(df2['n_targets'] == 0) & (df2['relative_fitness_mean'] > 0.9)]

# Sandbox

In [ ]:
# Find seed on a single gene

import re
# query_seed = 'AAAAA'
query_seed = 'AGACTTTGC'
# Find query_seed in a sequence
strand = 1
gene = 'guaB'
gene_seq = df_genes[df_genes["gene_name"] == gene].iloc[0]["sequence"]


list_seed_start_pos = []
list_seed_targets_coding_strand = []
list_spacers = []
# Check if it binds to the coding (non-template) strand.
# That is equivalent to being a subset of the non-coding (template) strand.
# So I should revcomp the sequence extracted from the gene
# This is desirable for better knockdown efficiency
template_seq = str(gene_seq.reverse_complement())
for m in re.finditer(query_seed+r'.GG', template_seq):
    seed_start = len(gene_seq) - m.end() + 1 + 3 # 1 for 0-based index, 3 for the PAM
    list_seed_start_pos.append(seed_start)
    list_seed_targets_coding_strand.append(True)
    list_spacers.append(gene_seq[seed_start-1:seed_start+20-1].reverse_complement())
# Check if it binds to the non-coding (template) strand.
# That is equivalent to being a subset of the coding (non-template) strand.
template_seq = str(gene_seq)
for m in re.finditer(query_seed+r'.GG', template_seq):
    print(m.start(), m.end())
    seed_start = m.start() + 1 + 3 # 1 for 0-based index, 3 for the PAM
    list_seed_start_pos.append(seed_start)
    list_seed_targets_coding_strand.append(False)
    list_spacers.append(gene_seq[seed_start:seed_start+20-1])
print(list_seed_start_pos)
print(list_seed_targets_coding_strand)
print(list_spacers)

# TODO Account for edge case where the seed is at the beginning or end of the gene
# There should always be room for a full spacer sequence